# Time Series Data Augmentation and Classification

We create a simple synthetic time-series classification task:
- Class 0: noisy sine waves
- Class 1: noisy square-like waves

A/B test:
- baseline
- augmented with jitter and scaling

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)

In [ ]:
def make_series_dataset(n_samples=3000, length=100):
    X, y = [], []
    t = np.linspace(0, 2*np.pi, length)
    for i in range(n_samples):
        label = i % 2
        if label == 0:
            x = np.sin(t) + np.random.normal(0, 0.1, size=length)
        else:
            x = np.sign(np.sin(t)) + np.random.normal(0, 0.1, size=length)
        X.append(x)
        y.append(label)
    X = np.array(X, dtype=np.float32)[..., None]
    y = np.array(y, dtype=np.int32)
    return X, y

X_train, y_train = make_series_dataset(2500)
X_test, y_test = make_series_dataset(800)

def augment_timeseries(X):
    X_aug = X.copy()
    jitter = np.random.normal(0, 0.05, size=X_aug.shape).astype(np.float32)
    scale = np.random.uniform(0.9, 1.1, size=(X_aug.shape[0], 1, 1)).astype(np.float32)
    return X_aug * scale + jitter

X_train_aug = np.concatenate([X_train, augment_timeseries(X_train[:1200])], axis=0)
y_train_aug = np.concatenate([y_train, y_train[:1200]], axis=0)

In [ ]:
def build_ts_model():
    model = keras.Sequential([
        layers.Input(shape=(100, 1)),
        layers.Conv1D(32, 5, activation="relu"),
        layers.MaxPooling1D(),
        layers.Conv1D(64, 3, activation="relu"),
        layers.GlobalAveragePooling1D(),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

base_model = build_ts_model()
base_model.fit(X_train, y_train, epochs=5, batch_size=64, verbose=0)
base_acc = base_model.evaluate(X_test, y_test, verbose=0)[1]

aug_model = build_ts_model()
aug_model.fit(X_train_aug, y_train_aug, epochs=5, batch_size=64, verbose=0)
aug_acc = aug_model.evaluate(X_test, y_test, verbose=0)[1]

print("Baseline accuracy:", round(base_acc, 4))
print("Augmented accuracy:", round(aug_acc, 4))